In [7]:
import pandas as pd
import numpy as np
import torch
from torch import nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import sklearn
from sklearn.metrics import classification_report
import time
import datetime
import os

In [8]:
#Loading Dataset
training_dataset = pd.read_csv('mitbih_train.csv', header = None, sep = ',')
testing_dataset = pd.read_csv('mitbih_test.csv', header = None, sep = ',')

X_train = training_dataset.iloc[:, :-1].values.astype(np.float32)
y_train = training_dataset.iloc[:, -1].values.astype(int).ravel()
X_test  = testing_dataset.iloc[:, :-1].values.astype(np.float32)
y_test  = testing_dataset.iloc[:, -1].values.astype(int).ravel()

In [9]:
#Store samples and corresponding layers
class MyDataset(Dataset):
    def __init__(self, X, y):
        self.data = torch.tensor(X, dtype = torch.float32).unsqueeze(1)
        self.labels = torch.tensor(y, dtype = torch.long)
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]
    
traindl = DataLoader(MyDataset(X_train, y_train), batch_size = 256, shuffle = True)
testdl = DataLoader(MyDataset(X_test, y_test), batch_size = 256)

In [10]:
#Define neural network architecture

class ECGClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv1d(1,32, kernel_size= 5, padding= 2),
            nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),
            
            nn.Conv1d(32,64, kernel_size= 5, padding= 2),
            nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),

            nn.Conv1d(64,128, kernel_size= 3, padding= 1),
            nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2),

            nn.Flatten(),
            nn.Linear(128*23, 256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, 5)

        )
    def forward(self,x):
        return self.layers(x)

In [11]:
#Train Neural Network on the Dataset

#GPU Acceleration
device = torch.device('cuda' if torch.cuda.is_available()else'cpu')
print(f'Using device: {device}')
model = ECGClassifier().to(device) # initializes model
weights   = torch.tensor(1.0 / np.bincount(y_train), dtype=torch.float32)
loss_function = nn.CrossEntropyLoss(weight=(weights / weights.sum()).to(device)) #for classification
optimizer = optim.Adam(model.parameters(), lr=1e-3) # simple Adam optimizer

for epoch in range(30):
    model.train()
    for xb, yb in traindl:
        loss = loss_function(model(xb.to(device)), yb.to(device))
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1}  loss: {loss.item():.4f}')

Using device: cpu
Epoch 5  loss: 0.0126
Epoch 10  loss: 0.0001
Epoch 15  loss: 0.0002
Epoch 20  loss: 0.0000
Epoch 25  loss: 0.0005
Epoch 30  loss: 0.0001


In [12]:
#Evaluation Mode

model.eval() #turns off dropout
preds, true = [], []
with torch.no_grad():
    for xb,yb in testdl:
        preds.extend(model(xb.to(device)).argmax(1).cpu())
        true.extend(yb)

print(classification_report(true, preds, target_names= ['N','S','V','F','Q']))

              precision    recall  f1-score   support

           N       0.99      0.98      0.99     18118
           S       0.71      0.85      0.78       556
           V       0.93      0.96      0.95      1448
           F       0.51      0.91      0.65       162
           Q       0.99      0.99      0.99      1608

    accuracy                           0.97     21892
   macro avg       0.83      0.94      0.87     21892
weighted avg       0.98      0.97      0.98     21892

